# Lecture 12 - Pseudorandom Numbers and Monte Carlo Methods

Last lecture, we discussed one technique to fit a model to a dataset, using a combination of linear algebra solvers and root finding methods. These methods underlie nonlinear least squares algorithms like `optimize.leastsq`, and are great ways to calculate the best-fit parameters for a model with only a few parameters to a small dataset. However, as the number of free parameters of your model increases, or the size and/or complexity of your dataset becomes larger, more sophisticated methods are needed. 

In today's lecture, we will be introducing the most widely used methodology to fit a model to a dataset: Markov-Chain Monte Carlo (MCMC). Before we discuss MCMC, we will first introduce the concept of pseudorandom numbers.

### Accepting and submitting

To accept this assignment on Classroom 50 with your GitHub profile enrolled in the class, click on the link \
**insert link** \
and follow the steps the webpage prompts.

To submit this assignment, follow the steps in "First time Classroom 50 setup" and "Accepting and Submitting Assignments" on the course webpage:\
https://psuastro410.github.io/tips/github/

### Resources and Acknowledgements

We thank Shane Davis for material that went into the design of this lecture.

# Pseudorandom Numbers

For many applications, it is useful to have random numbers, but it is nearly impossible for computer to generate *truly* random numbers because the generation is always based on some underlying algorithm.  This leads to the concept of **pseudorandom numbers**, which are numbers generated by an algorithm in a manner that attempts to minimize correlations across sequences of random numbers.

Most algorithms for generating (pseudo)random numbers require a seed to start the sequence. If one uses the same seed, the sequences of numbers generated will be identical. Below is a list of different random number generators: \
https://en.wikipedia.org/wiki/List_of_random_number_generators#Pseudorandom_number_generators_(PRNGs) \
For some applications (e.g. non-linear dynamics), the random number generator matters. For today, simple ones will work just fine.

A simple example of random number generator, that starts with a number $I_{\rm old} \in [0, 1)$, and outputs a new number $I_{\rm new} \in [0,1)$ with roughly uniform probability, is the Lehmer random number generator
$$
I_{\rm new} = (aI_{\rm old}\ {\rm mod}\ M)/M
$$
with $a=7^5$ and $M=2^{31} - 1$:

In [ ]:
def lehmer(u):
    a = 7**5            # 16807
    M = 2**31 - 1
    i_old = round(u * M)            # map (0,1) back to an integer
    i_new = (a * i_old) % M
    return i_new / M                # map back to (0,1)

Below we try this algorithm out a few times:

In [ ]:
u = 0.5
for _ in range(5):
    u = lehmer(u)
    print(u)

Usually we want something more sophisticated, which there are ready-made wrappers in python for.

## Random Number Generators in Python

Python provides the built-in `random` module and the `numpy.random` module for generating random numbers. This lecture will focus on `numpy.random`.

We start by generating random integers between 0 and 100:

In [ ]:
import numpy as np

# Lets start by drawing random intengers
# the following draws 5 integers
# between 0, 100 each time called
print(np.random.randint(0, 100, 5))
print(np.random.randint(0, 100, 5))
print(np.random.randint(0, 100, 5))

Notice that we got different random numbers each time. 

What if we wanted to be able to debug a calculation and required the same random numbers each time? Then we set a seed value:

In [ ]:
# set the seed value
np.random.seed(42)
print(np.random.randint(0, 100, 5))
print(np.random.randint(0, 100, 5))
print(np.random.randint(0, 100, 5))
# now reset the seed
np.random.seed(42)
print(np.random.randint(0, 100, 5))

If we want floats instead of integers, we use `random()` instead of `randint()`.  Note this will return numbers between 0 and 1.

In [ ]:
print(np.random.random(10))

To get a different range, we multiply by the desired range and add or subtract the minimum value.  Alternatively, we can use the `uniform()` function. For example, if we wan numbers between 1-3 we could do either of the following:

In [ ]:
print(np.random.uniform(1, 3, 20))
print(np.random.random(20)*2+1)

The module also provide function for drawing non-uniform distributions. For example, say we want to model the exponential distribution

In [ ]:
import matplotlib.pyplot as plt

y = np.random.standard_exponential(10000)

plt.hist(y, bins=30, edgecolor='black')
plt.yscale('log')
plt.show()

The exponential should show up as a straight line in the log plot so this gives the expected result.  NumPy provides a large number of commonly needed distributions.  A list can be found here:

https://numpy.org/doc/stable/reference/random/legacy.html

Let's say we want a Gaussian.

In [ ]:
mu = 0,
sigma = 0.1
y = np.random.normal(mu, sigma, 100000)

plt.hist(y, bins=30, edgecolor='black')
#plt.yscale('log')
plt.show()

New implementations have more object oriented programming, where we create a random number generator (rng) object:

In [ ]:
# default_rng takes a seed as an argument
rng = np.random.default_rng()
print(rng)
# uncomment this for more information on
# randon number generator
#help(rng)

# this will yield numbers between
# 0 and 1
print(rng.random(10))

Note that `rng.random()` corresponds closely to the legacy format function `random.random()` above. The names for drawing random numbers from different distributions are generally consistent with one another. The new interface grants more access to the internals of the random number generator, and does cool objected-oriented programming stuff, but there is not much difference if you just want some random numbers.

## Non-uniform Random number Generation

For many applications one wants random numbers that sample a distribution (e.g. gaussian) rather than uniform.  NumPy's `random` module supports many different distributions.  But, what do you do if you need a distribution that is not provided?  And how was the `random` module providing number from those distributions?

There are numerous algorithms for generating and transforming random numbers, but we will focus on the two most versatile methods: the inverse transform method and the rejection method.

### Inverse transform method

Lets example assume you have a normalized probability distribution $p(x)$ defined on the interval $[0, \infty)$.  Normalized means that
$$
\int_0^\infty p(x) dx = 1
$$
We can talk about the cumulative probability:
$$
P(x)=\int_0^x p(x') dx'.
$$
Here, $P(x)$ runs from 0 to 1, just like our uniform random number.  If we draw a uniform random number, we can then associate it with the cumulative probaiblity.  When we can perform the integral in the above equation to obtain an analytic function whose inverse is known, we can use the inverse transform method.

Let's asume you can generate a random number uniformly distributed between [0,1).  But we want a distribution from $p(y)$, where $p$ is some desired distribution.  Then we want 
$$
\int_{y_{\rm low}}^{y(x)} p(y') dy' = x
$$
Integrating the left hand side we have,
$$
P(y) - P(y_{\rm low}) = x
$$
where $P$ is the integral of $p$.  This gives a mapping between x and y, which is selected from the distribution $p$.

We need to be able to compute the inverse of the function $P$, which we denote $P^{-1}$. Then we can solve for $y$ in terms of $x$, our uniform random number, via
$$
y = P^{-1}(x+P(y_{\rm low}))
$$

**Example:**
Lets consider an exponential distribution
$$
p(y) = \exp(-y) \text{ for } y=[0,\infty].
$$
Hence we have that $y_{\rm low}=0$. Integrating $\exp(-y)$, we find that 
$$
P(y) - P(y_{\rm low}) = 1 - \exp(-y) = x 
$$
Inverting the exponential function,
$$
y = -\log(1-x).
$$

Lets try it out:

In [ ]:
rng = np.random.default_rng()
r = rng.random(100000)

y = -np.log(1-r) 

plt.hist(y, bins=30, edgecolor='black')
plt.yscale('log')
plt.show()

### Rejection method

What the integral does not yield an analytic function, or the inverse of that function is not easy to compute?

The rejection method is a powerful method that can be used to model almost any probability distribution, and relies on the assumption that we can sample a function that lies above the target distribution.

**Example:** We want to model a distribution between 0, $\pi$ that follows $\sin(x)$.  To make a normalized distribution,
$$
p(x) = \frac{\sin x}{2},
$$
which has a maximum of 0.5 on the interval.  

We then draw two random numbers: $r_1$ and $r_2$.  Let $r_1$ lie uniformly between $[0,\pi)$, and $r_2$ be uniform between $[0, 1)$.  We then evaluate $p(r_1)$.  If $r_2 < p(r_1)$ we take $r_1$ and added it to our sampled numbers.  Alternatively, if $r_2 > p(r_1)$ we reject it and draw **both** $r_1$ and $r_2$ again.

Lets first do this, say, 30 times, to get a feel of how this algorithm is working:

In [ ]:
x = np.linspace(0., np.pi, num=100)
p = lambda x: np.sin(x)/2

r1 = rng.random(30)*np.pi
r2 = rng.random(30)*0.5

sam_pass = np.where(p(r1) > r2)
sam_rej = np.where(p(r1) < r2)
plt.plot(x, p(x), 'k-')
plt.plot(r1[sam_pass], r2[sam_pass], 'g.', label='pass')
plt.plot(r1[sam_rej], r2[sam_rej], 'r.', label='reject')
plt.xlabel('$r_1$')
plt.ylabel('$r_2$')
plt.legend()
plt.show()

Ok, our algorithm is including more numbers near where $p(x)$ is highest, or has the highest probability. Conversly, it rejects numbers where the probability is lowest. We are crudely re-creating a sample of points where we expect to have the most points.

Lets see what our distribution looks like after doing this $10^5$ times:

In [ ]:
r1 = rng.random(10**5)*np.pi
r2 = rng.random(10**5)*0.5

y = r1[np.where(p(r1) > r2)]

plt.hist(y, bins=30, facecolor='green', edgecolor='black')
plt.show()

print("Efficiency: ",y.size/r1.size)

This seems to have worked. Note that we have evaluated the efficiency as the fraction of the samples that were rejected, which is a ratio of the area under our target curve $y=\sin x$, divided by the area under our sampling curve, $y=1$ over the sampled interval $[0,\pi)$.  The closer the sampling curve is our target curve, the more efficient our algorithm is.

**Note:** For multiple dimensions, the probability distribution depends jointly on two independent variables.

# 2. Monte Carlo Methods

Monte Carlo methods refers collectively to a somewhat diverse set of algorithms that utilize pseudorandom numbers to model probability distributions.  The the name Monte Carlo refers to a casio in Monaco, and was coined by physicist Stanislaw Ulam as a codename for work he was doing durin the Manhattan project.

In physics, Monte Carlo methods are used to solve problems in transport, kinematics, and statistical mechanics.  They are particularly well suited for algorithms that require integrating or sampling high dimensional spaces, which are not easily accessible with other conventional methods. Wikipedia provides an introduction to the methods here:

https://en.wikipedia.org/wiki/Monte_Carlo_method

## Monte Carlo Integration

Monte Carlo methods are used when a function is difficult to integrate analytically, difficult to compute using normal quadrature, and/or has high dimensionality.  For an example, consider the integrand
$$
I = \int_0^1\sin^2\left(\frac 1 x\right)dx 
$$
This integrand has a complex structure near $x=0$

In [ ]:
step=1.e-5
x_plot = np.arange(step, 1+step, step)
plt.plot(x_plot, np.sin(1/x_plot)**2)
plt.show()

How should we numerically integrate this? If we try to numerically integrate this with e.g. the simpson method, we find that it is extraordinarily sensitive to the way the numerical integral is done. 

Lets try Monte Carlo methods. First, we determine a bounding box around the integral and then pick a random x and y in this bound.  If $y < \sin^2(1/x)$, then this is part of the integral and should be counted.

Lets see what this looks like first with $10^3$ evaluations:

In [ ]:
logN = 4
N = 10**logN
x = np.random.rand(N)
y = np.random.rand(N)

int_in = y < np.sin(1/x)**2
int_out = y > np.sin(1/x)**2

fig, ax = plt.subplots(figsize=(10,5))
ax.plot(x[int_in], y[int_in], color='green', marker='.', ls='', label='in')
ax.plot(x[int_out], y[int_out], color='red', marker='.', ls='', label='out')
ax.plot(x_plot, np.sin(1/x_plot)**2, color='C0')
ax.legend()
plt.show()

The integral is then equal to the fraction that part of the integrand inside this bounding box:
$$
I \approx \frac{N_{\rm in}}{N}
$$

In [ ]:
for lgN in [4, 5, 6, 7, 8]: 
    # generate N random numbers
  N = int(10.**lgN)
  x = np.random.rand(N) # 0 to 1
  y = np.random.rand(N)

  integral = y[y < np.sin(1/x)**2].size/N
  print("N = {0} integral = {1:.6e}".format(N, integral))

The correct result is $\simeq 0.6734$.  The monte carlo integration is not bad, but it converges like $N^{-1/2}$ unlike other methods which generally converge at least as fast as $N^{-1}$, and so it can be comparitively slow for large $N$.

Monte Carlo is especially useful for multidimensional integrals.  Consider the integral 
$$
I = \int \exp(-x^2-y^2) dx dy
$$
We can compute this analytical to find 
$$
I = \int \exp(-x^2-y^2) dx dy = \int \exp(-r^2) 2\pi r dr = \pi
$$
In principle we would have to do an integral over the entire plane or transfrom variables, but since we know that the integral will fall off rapidly at large $x$ and $y$, we can truncate the range used for integral to keep things simple.

In [ ]:
box_size = 10
for lgN in [5, 6, 7, 8]: 
    N = int(1e1**lgN)
    x = box_size*(2*np.random.rand(N) - 1)# generate N random numbers
    y = box_size*(2*np.random.rand(N) - 1)
    z = np.random.rand(N)
    
    integral = 4*box_size**2*z[z<np.exp(-x*x - y*y)].size/N
    print("N = {0} integral = {1:.6e}".format(N, integral))

In addition to integration, there are a large number of ways in which Monte Carlo methods are used to simulate random walks. These can be walks in some physical space or some state space (e.g. thermodynamic ensemble). 

## 3. Markov Chain Monte Carlo

As mentioned above, Monte Carlo methods are often used to simualte randon walk problems.  A **Markov Chain** is a particular type of random walk that describes a sequence of steps in which the probability of each step depends only on the state of the system at the end of the previous step.

Markov chain Monte Carlo methods are used in statistical physics to estimate thermodynamic properties of systems. The classic example is the Metropolis algorithm, which e.g. is employed this ground breaking paper on ferrofluids: https://ui.adsabs.harvard.edu/abs/1999PhRvE..59.2424D/abstract

More generally, Markov Chain Monte Carlo algorithms are usually used to sample probability distributions.  Unlike the simple integrations we did above, the individual points or samples are not chosen independently of each other, but instead are chosen to better sample the function in locations that will contribute significantly to the integral. Markov Chain Monte Carlo is often referred to by its abbreviation MCMC, which we will follow from here on out.

In Bayesian statistics, MCMC methods are commonly used to calculate moments and credible intervals of posterior probability distributions. This method is now widely used for statistical modeling of data and there multiple Python packages to facilitate it. We will focus on this application for the remainder of the notebook.

### Model Fitting

Usually in this case, we are concerned about the computation of the posterior distribution function $\pi(\vec{\theta};\vec{x})$ of parameters $\vec{\theta}$ given some data $\vec{x}$ and a prior $\pi(\vec{\theta})$. We use boldface to denote that the quantities of interest can be treated as vectors.

We need to make use of Bayes' theorem
$$
\pi(\vec{\theta};\vec{x}) = \frac{\pi(\vec{\theta})p(\vec{x};\vec{\theta})}{p(\vec{x})}
$$
where $p(\vec{x};\vec{\theta})$ is a likelihood function and $p(\vec{x}) = \int \pi(\vec{\theta})p(\vec{x};\vec{\theta})d\vec{\theta}$ is essentially the normalization.

In essence, we want to compute
$$
p(\vec{x}) = \int \pi(\vec{\theta})p(\vec{x};\vec{\theta})d\vec{\theta}
$$
The key is that we only need to compute over the values of $\theta$ that really contribute.  So we can define a Metropolis algorithm that attmept to do this with the following steps:
* Begin with some parameter set $\vec{\theta}_1$
* Compute the joint likelihood $L_1 = \pi(\vec{\theta}_1)p(\vec{x};\vec{\theta}_1)$
* Pick a random proposal $\vec{\theta}_2$ from a distribution based on the priors.
* Compute the likelihood $L_2 = \pi(\vec{\theta}_2)p(\vec{x};\vec{\theta}_2)$
* Compute the transition probability $\rho = L_2/L_1$
* Accept the proposal with probability $\rho$

Lets consider an example to better understand what we mean. We generate a sinusoidal function with noise on top of it, similar to what we did previously when we first explored model fitting.

In [ ]:
# model function
def model(t, theta): 
    return theta[0]*np.sin(t + theta[1]) 
    
# function to generate our synthetic data
def generate_data(func, par, sigma=1):
  
    t = np.arange(-2*np.pi, 2*np.pi, 0.025*np.pi)
    #y = A*np.sin(theta + phi) + np.random.normal(0, sigma, theta.size)
    y = func(t, par) + np.random.normal(0, sigma, t.size)

    return t, y, np.array(par)

Lets plot the data with a contour plot:

In [ ]:
t_data, y_data, theta0 = generate_data(model, [5, np.pi/4])
fig, ax = plt.subplots()
ax.errorbar(t_data, y_data, color='C0', yerr=1, ls='')
ax.plot(t_data, model(t_data, theta0), color='C1')
ax.set_xlabel('t')
ax.set_ylabel('y')
plt.show()

Our goal is to calculate the amplitude and phase of this periodic function.

Next, we will define a log-likelihood function
$$
\log p(\vec{y}; \vec{\theta}) = -\sum_i\left(\frac{y_i - \theta_0\sin(t_i + \theta_1)}{\sigma}\right)^2,
$$
where we set $\sigma = 1$ and $t$ is our independent variable.

In [ ]:
# likelihood function
def loglikelihood(func, t, y, theta, sigma=1): 
    return -(((y-func(t, theta))/sigma)**2).sum()

We also need to choose some priors. Let's set $\theta_0$ to be a flat distribution between 1 and 10 and $\theta_1$ to be a flat distribution between 0 and $2\pi$.

In [ ]:
# prior probabilities
def priors(theta): 
    if(theta[0] < 1 or theta[0] > 10) : 
        return 0
    if(theta[1] < 0 or theta[1] > 2*np.pi) : 
        return 0
    return 1  

Finally, we need to implement the actual MCM algorithm described above. We  `move()` function that handles each step in the chain.

When evaluating whether or not to accept a move, we evaluate the relative probabilities of the moves:
1. If the new state is more likely (given the likelihood function), then we always accept the move.
2. If the new move is less likely, we draw a random number to see whether we should accept this move.

This allows us to explore the parameters space, while staying in regions of relative high probability. This is analogous to the rejection method, using out posterior distribution.

In [ ]:
def move(func, t, y, theta, h=0.1, TINY=1e-30):

    # This is the random move for your parameter, called the kernel. We will use a simple one of a gaussian ball
    dtheta = (2*np.random.rand(theta.size)-1.)*h

    thetanew = theta + dtheta
    # compute the priors
    p1 = priors(theta) 
    p2 = priors(thetanew)

    if p2 <= 0: 
        return theta, False
  
    p = p2/(p1+TINY)

    # compute the likelihoods
    logL1 = loglikelihood(func, t, y, theta)
    logL2 = loglikelihood(func, t, y, thetanew)

    # Probability to accept new step
    rho = min(np.exp(logL2 - logL1)*p, 1)
    
    # accept the move 
    if(np.random.rand() < rho):
        return thetanew, True

    return theta, False 

We next create a function `mcmc()` that runs a single Markov chain. This random walks over our parameters $\vec{\theta}$, to reproduce where the parameters have their most likely values, called the posteriors.

In addition, most MCMC chains have a "burn in" period. This is to anticipate before your random walk starts to spend times in regions with the highest probability, it will start in a region a bit far away. You throw out these initial steps in your MCMC algorithm.

In [ ]:
def mcmc(func, t, y, theta, burn_in=1000, MAX_CHAIN=500000):
    thetachain = [] 
    i = 0

    for _ in range(MAX_CHAIN): 
        theta, accepted_move = move(func, t, y, theta)
        if accepted_move:
            i += 1 
            if i > burn_in: 
                thetachain.append(theta)
                
    return np.array(thetachain)

Lets take a look at what the MCMC algorithm is doing. Lets to the first 300 steps, and plot the positions of the parameters $\vec{\theta}_i$, compared to the "true" parameters $\vec{\theta}_0$. We start with an intial guess, and let the MCMC do its thing:

In [ ]:
guess = np.zeros(2)
guess[0] = 2
guess[1] = 0

thetachain = mcmc(model, t_data, y_data, guess, burn_in=0, MAX_CHAIN=300)

plt.plot(thetachain[:,0], thetachain[:,1])
plt.plot(theta0[0], theta0[1], 'ks')
plt.xlabel(r'$\theta$[0]')
plt.ylabel(r'$\theta$[1]')
plt.show()

We see the MCMC wanders until $\vec{\theta}_i$ gets close to the true values of $\vec{\theta}_0$. The walker then causes the $\vec{\theta}_i$ values to "cluster" near $\vec{\theta}_0$. This clustering traces out the most likely values for $\vec{\theta}_0$, called the "posterior". 

In addition, most MCMC chains have a "burn in" period. You can see this as the initial "wandering" before $\vec{\theta}_i$ gets close to the true values of $\vec{\theta}_0$. Because this part of the chain does not estimate the posterior, it is common practice to just chop these values off.

Lets do this for a longer chain:

In [ ]:
guess = np.zeros(2)
guess[0] = 2
guess[1] = 0

thetachain = mcmc(model, t_data, y_data, guess, MAX_CHAIN=100000)
print("chain length = {2} MCMC mean = {0}, actual = {1}".format(
      str(np.average(thetachain, axis=0)), str(theta0), thetachain.shape[0]))

Now that we have implemented the algorithm, and have the chain of values of $\vec{\theta}$. Consider the computation of the expectation values of the parameters $\vec{\theta}$ over the distribution given by the posterior distribution, $\pi(\vec{\theta};\vec{x})$ or
$$
\left<\vec{\theta}\right>_{\pi} = \int \vec{\theta}\pi(\vec{\theta};\vec{x}) d\vec{\theta} \approx \sum_i \vec{\theta}_i\pi(\vec{\theta_i};\vec{x}) \Delta\vec{\theta}_i
$$
Here, $\vec{\theta}_i$ can be the saved values of your parameters during your random walk, and $\Delta \vec{\theta}_i$ is the spacing of your numerical grid.

An important point is that when one takes the average over the entire Markov chain, then the values of the parameters should reduce to the expected value. If, however, one takes the distribution of the parameters, i.e., take a histogram of the values of the parameters in the chain, the distribution of the parameters is the posterior distribution.  This is how these parameters distributions for various models are usually generated. 

We do this below. We generate an initial guess, run our MCMC, and get best-fit parameter values. To generate the posteriors, we use python's histogram features to generate the grid spacing, with our already saved $\vec{\theta}_i$ values from the chain:

In [ ]:
# Now plot the 2D histogram
plt.hist2d(thetachain[:,0], thetachain[:,1], bins=100)
plt.xlabel("Amplitude")
plt.ylabel("Phase")
plt.show()

In [ ]:
# And the 1D histograms
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(8, 5))
axes[0].hist(thetachain[:,0], bins=30)
axes[1].hist(thetachain[:,1], bins=30)
axes[0].set_xlabel(r"Amplitude $\theta$[0]")
axes[1].set_xlabel(r"Phase $\theta$[1]")
plt.show()

# Exercises (10 pt)

## Exercise 1 - Muddiest Moment (2 pt)

From today's lecture, which concept did you find the most confusing?

**Answer below**


## Exercise 2 - Re-run exercises 1, 3, 4, 5 from Lecture 11 (2 pt)

In the code cells below, copy and paste your answers for Exercises 1, 3, 4, 5, from Lecture 11. 

In [ ]:
## Lecture 11, Exercise 1


In [ ]:
## Lecture 11, Exercise 3 and 4


In [ ]:
## Lecture 11, Exercise 5


## Exercise 3 - Write the priors for $H_0$ and $c$ (2 pt)

Write prior functions for $\vec{\theta} =  (H_0, c)$. Force $H_0$ to lie between $1$-$10^3$ km/s/Mpc, and $c$ to lie between $10^2$-$10^7$ km/s.

In [ ]:
## Answer here


## Exercise 4 - Log Probability for $H_0$ and $c$ (2 pt)

In terms of the $\chi$ from last lecture, the log probability is
$$
\log p = -\chi^2
$$
Write a function that calculates the log probability for $H_0$ and $c$ in terms of the residual function of Exercise 2.

In [ ]:
## Answer here


## Exercise 5 - Calculate posteriors for $H_0$ and $c$

Using the MCMC code above, calculate the posteriors for $H_0$ and $c$ in the cell below, and print the histogram.

In [ ]:
## Answer here
